In [ ]:
# Section 1: 安装依赖

!pip install -U transformers datasets trl peft accelerate

In [87]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [88]:
# Section: 创建保存目录

import os
from datetime import datetime

PROJECT_NAME = "tinyllama_home_lora_2"
RUN_TAG = "intent_stage_v2"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_NAME = f"{RUN_TAG}_{timestamp}"

BASE_OUTPUT_DIR = f"./{PROJECT_NAME}"
RUN_DIR = os.path.join(BASE_OUTPUT_DIR, RUN_NAME)
CHECKPOINT_DIR = os.path.join(RUN_DIR, "checkpoints")
FINAL_ADAPTER_DIR = os.path.join(RUN_DIR, "final_adapter")

os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)

print("RUN_DIR =", RUN_DIR)
print("CHECKPOINT_DIR =", CHECKPOINT_DIR)
print("FINAL_ADAPTER_DIR =", FINAL_ADAPTER_DIR)

RUN_DIR = ./tinyllama_home_lora_2\intent_stage_v2_20260420_174208
CHECKPOINT_DIR = ./tinyllama_home_lora_2\intent_stage_v2_20260420_174208\checkpoints
FINAL_ADAPTER_DIR = ./tinyllama_home_lora_2\intent_stage_v2_20260420_174208\final_adapter


In [1]:
# Section 2: 导入依赖

import os
import json
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

In [2]:
# Section 3: 基础配置

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

TRAIN_FILE = "train.jsonl"
VAL_FILE = "val.jsonl"

OUTPUT_DIR = "./tinyllama_home_lora"
FINAL_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "final_adapter")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("MODEL_NAME =", MODEL_NAME)
print("TRAIN_FILE =", TRAIN_FILE)
print("VAL_FILE =", VAL_FILE)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("FINAL_ADAPTER_DIR =", FINAL_ADAPTER_DIR)
print("DEVICE =", DEVICE)

MODEL_NAME = TinyLlama/TinyLlama-1.1B-Chat-v1.0
TRAIN_FILE = train.jsonl
VAL_FILE = val.jsonl
OUTPUT_DIR = ./tinyllama_home_lora
FINAL_ADAPTER_DIR = ./tinyllama_home_lora\final_adapter
DEVICE = cuda


In [89]:
# Section 4: 检查训练文件

assert os.path.exists(TRAIN_FILE), f"Training file not found: {TRAIN_FILE}"
assert os.path.exists(VAL_FILE), f"Validation file not found: {VAL_FILE}"

print("Training and validation files found.")

Training and validation files found.


In [90]:
# Section 5: 加载 tokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)

Tokenizer loaded.
Pad token: </s>


In [93]:
# Section 6: 加载数据集

dataset = load_dataset(
    "json",
    data_files={
        "train": TRAIN_FILE,
        "validation": VAL_FILE,
    }
)

print(dataset)
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 181
    })
    validation: Dataset({
        features: ['input', 'output'],
        num_rows: 31
    })
})
Train size: 181
Validation size: 31


In [ ]:
# Section 6.5: 补充缺失的 general_qa 和 invalid 训练样本
# Issue 6 fix: 原始训练数据只包含 direct_command 和 needs_clarification，
# 缺少 general_qa 和 invalid 类型，导致模型无法可靠输出这两类。

import json

EXTRA_TRAIN_EXAMPLES = [
    {"input": "Nova, how do I eat an apple?", "output": '{"type":"general_qa","answer":"Wash it first, then eat it."}'},
    {"input": "Nova, what time is it?", "output": '{"type":"general_qa","answer":"I don\'t have access to a clock, but you can check your phone."}'},
    {"input": "Nova, can I eat leftovers after 3 days?", "output": '{"type":"general_qa","answer":"Yes, most cooked food is safe for 3-4 days in the fridge."}'},
    {"input": "Nova, what's the weather like?", "output": '{"type":"general_qa","answer":"I don\'t have weather data, but you can check a weather app."}'},
    {"input": "Nova, how do I boil an egg?", "output": '{"type":"general_qa","answer":"Place the egg in boiling water for 6-12 minutes depending on your preference."}'},
    {"input": "Nova, what is 2 plus 2?", "output": '{"type":"general_qa","answer":"4."}'},
    {"input": "Hello.", "output": '{"type":"invalid"}'},
    {"input": "Never mind.", "output": '{"type":"invalid"}'},
    {"input": "Nova, nothing.", "output": '{"type":"invalid"}'},
    {"input": "Hmm.", "output": '{"type":"invalid"}'},
    {"input": "Okay thanks.", "output": '{"type":"invalid"}'},
    {"input": "Just testing.", "output": '{"type":"invalid"}'},
]

EXTRA_VAL_EXAMPLES = [
    {"input": "Nova, how long should I microwave rice?", "output": '{"type":"general_qa","answer":"About 2-3 minutes, stirring halfway through."}'},
    {"input": "Nova, is it going to rain today?", "output": '{"type":"general_qa","answer":"I don\'t have weather data, please check a weather app."}'},
    {"input": "Hi there.", "output": '{"type":"invalid"}'},
    {"input": "Forget it.", "output": '{"type":"invalid"}'},
]

# Append to train.jsonl
with open(TRAIN_FILE, "a", encoding="utf-8") as f:
    for ex in EXTRA_TRAIN_EXAMPLES:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

# Append to val.jsonl
with open(VAL_FILE, "a", encoding="utf-8") as f:
    for ex in EXTRA_VAL_EXAMPLES:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"Appended {len(EXTRA_TRAIN_EXAMPLES)} examples to {TRAIN_FILE}")
print(f"Appended {len(EXTRA_VAL_EXAMPLES)} examples to {VAL_FILE}")

In [94]:
# Section 7: 查看原始样本

print("Sample train row:")
print(dataset["train"][0])

Sample train row:
{'input': 'Nova, turn on the light.', 'output': '{"type":"direct_command","device":"light","action":"turn_on","value":null}'}


In [ ]:
SYSTEM_PROMPT = """
Return exactly one JSON object.

Allowed outputs:

{"type":"direct_command","device":"light|curtain|window|ac","action":"turn_on|turn_off|set_brightness|rgb_cycle|open|close|set_position|set_temperature","value":null_or_int}

{"type":"needs_clarification","question":"...","options":["...","..."]}

{"type":"general_qa","answer":"..."}

{"type":"invalid"}

Rules:
- If the user directly names a device and a clear action, always output direct_command.
- If the user makes an indirect or ambiguous home-control request, output needs_clarification.
- If the user asks a normal non-device question, output general_qa.
- If there is no meaningful request, output invalid.
- For the same kind of ambiguous request, keep the question and options consistent.
- Output JSON only.
- No extra text.
""".strip()

In [96]:
# Section 9: 格式化训练样本

def format_example(example):
    text = (
        f"<|system|>\n{SYSTEM_PROMPT}\n"
        f"<|user|>\n{example['input']}\n"
        f"<|assistant|>\n{example['output']}"
    )
    return {"text": text}

dataset = dataset.map(format_example)

Map:   0%|          | 0/181 [00:00<?, ? examples/s]

Map:   0%|          | 0/31 [00:00<?, ? examples/s]

In [97]:
# Section 10: 检查格式化后的样本

print("Formatted sample:")
print(dataset["train"][0]["text"])

Formatted sample:
<|system|>
Return exactly one JSON object.

Allowed outputs:

{"type":"direct_command","device":"light|curtain|window|ac","action":"turn_on|turn_off|set_brightness|rgb_cycle|open|close|set_position|set_temperature","value":null_or_int}

{"type":"needs_clarification","question":"...","options":["...","..."]}

Rules:
- If the user directly names a device and a clear action, always output direct_command.
- If the user makes an indirect or ambiguous home-control request, output needs_clarification.
- For the same kind of ambiguous request, keep the question and options consistent.
- Output JSON only.
- No extra text.
<|user|>
Nova, turn on the light.
<|assistant|>
{"type":"direct_command","device":"light","action":"turn_on","value":null}


In [98]:
# Section 11: 加载 base model

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Base model loaded.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Base model loaded.


In [99]:
# Section 12: 配置 LoRA

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)

model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
# Section 13: 配置训练参数

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=5,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    bf16=False,
    report_to="none",
)

In [101]:
# Section 14: 构建 Trainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
)

print("Trainer ready.")

Adding EOS to train dataset:   0%|          | 0/181 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/181 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/31 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/31 [00:00<?, ? examples/s]

Trainer ready.


In [102]:
# Section 15: 开始训练

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss
1,1.631976,1.186830
2,0.611577,0.227528
3,0.130817,0.096006


TrainOutput(global_step=69, training_loss=0.878090040407319, metrics={'train_runtime': 182.0969, 'train_samples_per_second': 2.982, 'train_steps_per_second': 0.379, 'total_flos': 768178854199296.0, 'train_loss': 0.878090040407319})

In [103]:
# Section: checkpoint 保存配置

training_args = SFTConfig(
    output_dir=CHECKPOINT_DIR,   # checkpoint 保存目录
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",       # 每个 epoch 自动保存 checkpoint
    save_total_limit=2,          # 最多保留 2 个 checkpoint
    fp16=torch.cuda.is_available(),
    bf16=False,
    report_to="none",
)

In [104]:
def extract_first_json_object(text: str):
    start = text.find("{")
    if start == -1:
        return None

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]

    return None


def has_complete_json_object(text: str) -> bool:
    start = text.find("{")
    if start == -1:
        return False

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return True

    return False
from transformers import StoppingCriteria, StoppingCriteriaList

class JsonStopOnComplete(StoppingCriteria):
    def __init__(self, tokenizer, prompt_length: int):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores, **kwargs):
        generated_ids = input_ids[0][self.prompt_length:]
        text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        return has_complete_json_object(text)

In [105]:
# Section 17: 训练后简单推理测试

def run_inference(test_input: str, max_new_tokens: int = 80):
    prompt = (
        f"{SYSTEM_PROMPT}\n\n"
        f"Input: {test_input}\n"
        f"Output:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_length = inputs["input_ids"].shape[1]

    stopping_criteria = StoppingCriteriaList([
        JsonStopOnComplete(tokenizer, prompt_length)
    ])

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            stopping_criteria=stopping_criteria,
            use_cache=True,
        )

    generated_ids = outputs[0][prompt_length:]
    raw_output = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    json_str = extract_first_json_object(raw_output)

    return {
        "raw_output": raw_output,
        "first_json": json_str
    }

In [106]:
# Section 18: 测试样例

test_inputs = [
    "Nova, turn on the light.",
    "Nova, turn off the light.",
    "Nova, open the curtain.",
    "Nova, close the window.",
    "Nova, set the AC to 24 degrees.",
    "Nova, I feel a little bit cold.",
    "Nova, I feel cold.",
    "Nova, it's a bit dark.",
    "Nova, this room feels dark.",
    "Nova, fuck this light.",
    "Nova, this light is annoying.",
    "Nova, make this room lively."
]

for x in test_inputs:
    print("=" * 80)
    print("INPUT :", x)
    print("OUTPUT:", run_inference(x))

Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INPUT : Nova, turn on the light.


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


OUTPUT: {'raw_output': '{"type":"direct_command","device":"light","action":"turn_on","value":null}', 'first_json': '{"type":"direct_command","device":"light","action":"turn_on","value":null}'}
INPUT : Nova, turn off the light.


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

OUTPUT: {'raw_output': '{"type":"needs_clarification","question":"...","options":["...","..."]}', 'first_json': '{"type":"needs_clarification","question":"...","options":["...","..."]}'}
INPUT : Nova, fuck this light.


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


OUTPUT: {'raw_output': '{"type":"direct_command","device":"light","action":"turn_on","value":null}', 'first_json': '{"type":"direct_command","device":"light","action":"turn_on","value":null}'}
INPUT : Nova, this light is annoying.


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


OUTPUT: {'raw_output': '{"type":"direct_command","device":"light","action":"turn_off","value":null}', 'first_json': '{"type":"direct_command","device":"light","action":"turn_off","value":null}'}
INPUT : Nova, make this room lively.
OUTPUT: {'raw_output': '{"type":"direct_command","device":"light|curtain|window|ac","action":"set_brightness","value":5}', 'first_json': '{"type":"direct_command","device":"light|curtain|window|ac","action":"set_brightness","value":5}'}


In [ ]:
# Section: checkpoint 保存配置

training_args = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=5,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    bf16=False,
    report_to="none",
)

In [ ]:
# from transformers import AutoTokenizer, AutoModelForCausalLM
# from peft import PeftModel
# import torch

# MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# ADAPTER_PATH = "./tinyllama_home_lora/final_adapter"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
#     device_map="auto"
# )

# model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
# model.eval()